# **CLIP Adaptation Effect**

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

scores_dir = "/content/gdrive/MyDrive/Scores/Scores"

baseline_clip = pd.read_csv(f"{scores_dir}/baseline_CLIP.csv")
finetuned_clip = pd.read_csv(f"{scores_dir}/finetuned_CLIP.csv")
sota_clip = pd.read_csv(f"{scores_dir}/SOTA_CLIP.csv")

clip_df = baseline_clip.merge(finetuned_clip, on="Category").merge(sota_clip, on="Category")
assert len(clip_df) == 76, f"Expected 76 paired categories, found {len(clip_df)}"

# Each fine-tuned configuration is compared only with its own unadapted base model.
comparisons = {
    "Mitsua": ("MITSUA_CLIP", {"DreamBooth": "DREAM_CLIP", "LoRA": "LORA_CLIP", "TI": "TI_CLIP"}),
    "SD2.1": ("SD_CLIP", {"DreamBooth": "SD21_dream", "LoRA": "SD21_lora", "TI": "SD21_ti"}),
    "JuggXL": ("JUGGXL_CLIP", {"DreamBooth": "JUGGXL_dream", "LoRA": "JUGGXL_lora", "TI": "JUGGXL_ti"}),
}

alpha = 0.05
n_tests = sum(len(methods) for _, methods in comparisons.values())
bonferroni_threshold = alpha / n_tests

wilcoxon_rows = []
for model, (base_col, methods) in comparisons.items():
    for method, tuned_col in methods.items():
        difference = clip_df[tuned_col] - clip_df[base_col]
        statistic, p_value = wilcoxon(clip_df[tuned_col], clip_df[base_col])
        wilcoxon_rows.append({
            "Foundation model": model,
            "Method": method,
            "Base median": clip_df[base_col].median(),
            "Fine-tuned median": clip_df[tuned_col].median(),
            "Median difference": difference.median(),
            "Categories improved": int((difference > 0).sum()),
            "W": statistic,
            "p-value": p_value,
            "Significant": p_value < bonferroni_threshold,
        })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)

print(f"Paired Wilcoxon signed-rank tests across {len(clip_df)} categories")
print(f"Bonferroni threshold: {alpha}/{n_tests} = {bonferroni_threshold:.4f}")
wilcoxon_df